# Determinism, Noise Models, and Replay

This notebook covers three practical topics:
- Reproducibility controls (`random_seed`, `shot_offset`, `shot_increment`)
- Noise injection with `DepolarizingErrorModel`
- Classical replay simulation for debugging program flow
- Quantum replay simulation for debugging quantum results

In [2]:
from guppylang.decorator import guppy
from guppylang.std.quantum import qubit, h, cx, measure
from guppylang.std.builtins import result

from selene_sim.build import build
from selene_sim import Stim, Quest, DepolarizingErrorModel, ClassicalReplay, QuantumReplay

In [3]:
@guppy
def bell() -> None:
    q0 = qubit()
    q1 = qubit()
    h(q0)
    cx(q0, q1)
    result("c0", measure(q0))
    result("c1", measure(q1))

runner = build(bell.compile(), "bell")

# 1. Reproducible shot streams

With fixed seeds, repeating a run reproduces the same sequence. Except for in the cases of announced breaking changes, simulations are intended to be reproducible across Selene versions.

They should also be reproducible across different operating systems. A notable exception is Quest, which defines its random number generation in a manner that differs on Windows compared with MacOS and Linux.

In [4]:
sim = Stim(random_seed=12345)

full_results = [dict(shot) for shot in runner.run_shots(sim, n_qubits=2, n_shots=20)]
results_again = [dict(shot) for shot in runner.run_shots(sim, n_qubits=2, n_shots=20)]
reproduces_prior_results = (results_again == full_results)
print(f"Reproduces prior results: {reproduces_prior_results}")

known_results = [{'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 1, 'c1': 1}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0},
                 {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1},
                 {'c0': 1, 'c1': 1}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0},
                 {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 1, 'c1': 1}]

reproduces_known_results = (full_results == known_results)
print(f"Reproduces known results: {reproduces_known_results}")

Reproduces prior results: True
Reproduces known results: True


If you wish to reproduce a subset of shots, you can do so by passing a shot_offset and shot_increment to `run_shots`.

In this example, we wish to replay shot 3, 7, 11, 15, 19 from our previous run, we can pass:
- `shot_offset = 3`
- `shot_increment = 4`
- `n_shots=5`

In [5]:
subset = [
    dict(shot)
    for shot in runner.run_shots(
        sim,
        n_qubits=2,
        n_shots=5,
        shot_offset=3,
        shot_increment=4,
    )
]

print("subset matches full[3::4]:", subset == full_results[3::4])

subset matches full[3::4]: True


This approach may also be used for distributing shots over multiple machines.

# 2. Add noise with `DepolarizingErrorModel`

Set `p_init`, `p_meas`, `p_1q`, `p_2q` to control where depolarizing noise is injected.

In [6]:
error_model = DepolarizingErrorModel(
    random_seed=2026,
    p_init=0.01,
    p_meas=0.01,
    p_1q=0.01,
    p_2q=0.01,
)

ideal_shots = [dict(shot) for shot in runner.run_shots(Quest(random_seed=7), n_qubits=2, n_shots=100)]
noisy_shots = [
    dict(shot)
    for shot in runner.run_shots(
        Quest(random_seed=7),
        error_model=error_model,
        n_qubits=2,
        n_shots=100,
    )
]

for shot, (ideal, noisy) in enumerate(zip(ideal_shots, noisy_shots)):
    if ideal != noisy:
        print(f"Shot {shot} differs in the presence of noise:")
        print(f"   Ideal results: {ideal}")
        print(f"   Noisy results: {noisy}")

Shot 20 differs in the presence of noise:
   Ideal results: {'c0': 1, 'c1': 1}
   Noisy results: {'c0': 0, 'c1': 1}
Shot 34 differs in the presence of noise:
   Ideal results: {'c0': 0, 'c1': 0}
   Noisy results: {'c0': 1, 'c1': 0}
Shot 40 differs in the presence of noise:
   Ideal results: {'c0': 0, 'c1': 0}
   Noisy results: {'c0': 0, 'c1': 1}
Shot 41 differs in the presence of noise:
   Ideal results: {'c0': 1, 'c1': 1}
   Noisy results: {'c0': 1, 'c1': 0}
Shot 62 differs in the presence of noise:
   Ideal results: {'c0': 1, 'c1': 1}
   Noisy results: {'c0': 1, 'c1': 0}
Shot 79 differs in the presence of noise:
   Ideal results: {'c0': 1, 'c1': 1}
   Noisy results: {'c0': 0, 'c1': 1}


# 3. Classical replay

`ClassicalReplay` feeds predefined measurement sequences to validate control flow expectations.

In [7]:
replay_measurements = [[False, False], [True, True], [False, False], [True, True]]
classical_results = [
    dict(shot)
    for shot in runner.run_shots(
        ClassicalReplay(measurements=replay_measurements),
        n_qubits=2,
        n_shots=len(replay_measurements),
    )
]

print(classical_results)

[{'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}, {'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}]


It is primarily useful for validating assumptions about the flow of a program. For example, if a loop is conditioned on a measurement outcome, we can ensure that it has the behaviour we require:

In [26]:
@guppy
def repeat_until_true() -> None:
    q = qubit()
    h(q)
    c = measure(q)
    result("c", c)
    if not c:
        repeat_until_true()

loop_runner = build(repeat_until_true.compile())

All programs should terminate with the first True measurement

In [28]:
valid_measurements = [
    [True],
    [False, True],
    [False, False, False, False, False, True]
]
valid_replay = ClassicalReplay(measurements=valid_measurements)
valid_results = [
    list(shot)
    for shot in loop_runner.run_shots(
        simulator=valid_replay,
        n_qubits=10,
        n_shots=len(valid_measurements),
    )
]
print(valid_results)

[[('c', 1)], [('c', 0), ('c', 1)], [('c', 0), ('c', 0), ('c', 0), ('c', 0), ('c', 0), ('c', 1)]]


Whereas invalid measurement sequences passed to ClassicalReplay produce an error, perhaps providing insight into a failed axiom of your program:

In [34]:
invalid_measurements = [
    [],            # at least one measurement should happen
    [False],       # more measurements are expected
    [True, False], # the second measurement should not have taken place 
]
# as each shot will fail, we invoke them one by one
for invalid in invalid_measurements:
    invalid_replay = ClassicalReplay(measurements=[invalid])
    try:
        invalid_results = [
            list(shot)
            for shot in loop_runner.run_shots(
                simulator=invalid_replay,
                n_qubits=10,
                n_shots=1,
            )
        ]
    except Exception as e:
        print(f"Shot failed: {e}")

Shot failed: Panic (#100001): ErrorModelPlugin: handle_operations failed
----- stderr -----
Failed to measure qubit 0: All measurements have already been performed for shot 0.
Error: Failed to handle operations
SimulatorPlugin(Unknown): measure failed

------------------

Shot failed: Panic (#100001): ErrorModelPlugin: handle_operations failed
----- stderr -----
Failed to measure qubit 0: All measurements have already been performed for shot 0.
Error: Failed to handle operations
SimulatorPlugin(Unknown): measure failed

------------------

Shot failed: Panic (#100001): ErrorModelPlugin: shot_end failed
----- stderr -----
Error: Failed to end the current shot
Not all measurements have been performed for shot 0.
Error: Failed to end the current shot
SimulatorPlugin(Unknown): shot_end failed
[interface] error: Error ending shot 0: error code 100001

------------------



# 4. Quantum replay

`QuantumReplay` postselects to requested measurements, then optionally resumes physical measurement for the rest.

This provides two main uses:
- Selecting into a specific interesting (but not implausible) branch of your program before continuing as normal
- Validating that measurement sequences are possible under simulation

For example, in the bell state program we expect that the two measurements are in agreement. We can observe this by only providing a single measurement, and watching the second measure in agreement:

In [35]:
partial_replay = QuantumReplay(
    simulator=Quest(random_seed=111),
    resume_with_measurement=True,
    measurements=[[False], [True]],
)

quantum_results = [
    dict(shot)
    for shot in runner.run_shots(partial_replay, n_qubits=2, n_shots=2)
]

print(quantum_results)

[{'c0': 0, 'c1': 0}, {'c0': 1, 'c1': 1}]


Or we can provide incorrect measurement results and expect an error to occur:

In [39]:
partial_replay = QuantumReplay(
    simulator=Quest(random_seed=111),
    resume_with_measurement=True,
    measurements=[[False, True], [True, False]],
)

try:
    quantum_results = [
        dict(shot)
        for shot in runner.run_shots(partial_replay, n_qubits=2, n_shots=2)
    ]
except Exception as e:
    print(f"Error: {e}")

Error: Panic (#100001): ErrorModelPlugin: handle_operations failed
----- stderr -----
Error: Failed to postselect qubit
Postselection of 1 on qubit 1 is too unlikely to postselect. The probability of this outcome is 7.70e-33.
Failed to measure qubit 1: SimulatorPlugin(Unknown): postselect failed
Error: Failed to handle operations
SimulatorPlugin(Unknown): measure failed

------------------

